# From Waveform to Genre — Data preparation

### Data Preparation & Stratified Splitting

**FMA dataset (citation)**  
Defferrard, M., Benzi, K., Vandergheynst, P., & Bresson, X. (2017). *FMA: A Dataset for Music Analysis*. 18th ISMIR. PDF: https://arxiv.org/pdf/1612.01840.pdf

**License / data note:** FMA metadata is CC BY 4.0; audio files follow per-artist Creative Commons licenses. Consult the original dataset for terms of use.

---

### Project adaptation note
This notebook adapts the FMA data pipeline for the current workflow and interactive environments (Google Colab / Drive). Key adaptations include Colab/Drive path variables, idempotent download & extraction steps, automatic manifest generation for reproducibility, and helper utilities for checksums and seeding. Any reuse of original code or logic from the mdeff/fma project is indicated in the header and documented in the repository.

### Short comparison vs. original mdeff/fma

- Similarities:
  - Uses FMA metadata (tracks.csv), selects the "small" subset, and constructs stratified train/val/test splits.
  - Employs fixed SEED for reproducibility and verifies metadata checksums.

- Project-specific adaptations:
  - Colab/Drive integration: explicit `LOCAL_DATA` / `DRIVE_ROOT` variables and Drive sync utilities.
  - Idempotent pipeline: skip existing downloads and extractions to allow safe re-runs in interactive sessions.
  - Manifest generation: saves package versions, checksums and split sizes (improves reproducibility for this workflow).
  - Small API/wrapper changes: helper functions (from `utils.py`) and modified I/O conventions to match project structure.

- Conclusion:
  - The notebook follows the original pipeline logic but includes practical adjustments for Colab usage and reproducibility. It is appropriate to describe the repository as an adaptation of the mdeff/fma pipeline; details are documented in the repository.

## Mount Drive & setup paths

In [ ]:
# mount Google Drive into Colab
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
# Import the os module for filesystem operations
import os
# set the project working directory in the Colab filesystem
WORKDIR = "/content/waveform_genre_project"
# define a local data storage path inside the working directory
LOCAL_DATA = os.path.join(WORKDIR, "data_storage")
# define the Drive path where analysis outputs will be stored
DRIVE_ROOT = "/content/drive/MyDrive/waveform_analysis_outputs"
# ensure the local data directory exists (create if missing)
os.makedirs(LOCAL_DATA, exist_ok=True)
# ensure the Drive output directory exists (create if missing)
os.makedirs(DRIVE_ROOT, exist_ok=True)
# change the current process working directory to the project folder
os.chdir(WORKDIR)
# print the configured paths to verify everything is set up correctly
print("WORKDIR:", WORKDIR)
print("LOCAL_DATA:", LOCAL_DATA)
print("DRIVE_ROOT:", DRIVE_ROOT)

### Library Imports and Helper Functions

This code imports the necessary libraries and defines two utility functions: one for calculating a file's checksum and another for saving Python objects to JSON files.

In [ ]:
import hashlib, json
import urllib.request, zipfile
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# define small helper functions
# function to calculate the checksum (hash) of a file to verify its integrity
def checksum(path, algo="sha1"):
    # Initialize a hash object using the specified algorithm
    h = hashlib.new(algo)
    # open the file in binary read mode
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    # return the final hexadecimal digest
    return h.hexdigest()
# function to save a Python object (like a dict) to a formatted JSON file
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

Sets configuration variables: it points DATA_DIR to the local data folder, specifies the URL and expected SHA1 checksum of the FMA metadata archive, sets the number of tracks to use per genre to 100, and fixes the random seed to 42.

In [ ]:
DATA_DIR = LOCAL_DATA
METADATA_URL = "https://os.unil.cloud.switch.ch/fma/fma_metadata.zip"
METADATA_SHA1 = "f0df49ffe5f2a6008d7dc83c6915b31835dfe733"
TRACKS_PER_GENRE = 100
SEED = 42

Thе code ensures the metadata directory exists, downloads the FMA metadata ZIP if missing, and verifies its SHA-1 checksum before proceeding.

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
# build the full local path for the metadata ZIP file
zip_path = os.path.join(DATA_DIR, "fma_metadata.zip")
# If the ZIP is not already downloaded, fetch it from the metadata URL
if not os.path.exists(zip_path):
    print("Downloading metadata (~342 MB)...")
    urllib.request.urlretrieve(METADATA_URL, zip_path)
else:
    print("Metadata archive exists, reusing.")

print("Verifying checksum...")
# compute and assert the file's SHA-1 matches the expected value (raises error on mismatch)
assert checksum(zip_path, "sha1") == METADATA_SHA1, "Metadata checksum mismatch"
# confirm the archive integrity is valid
print("Checksum OK")